In [60]:
# CONFIG
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT

DATA_DIR = PROJECT_ROOT / "data"
LOG_DIR = PROJECT_ROOT / "logs"
MODEL_PATH = PROJECT_ROOT / "models" / "lenet_mnist.pt"

SEED = 42

TRAIN_RATIO = 0.9

BATCH_SIZE = 256

NUM_EPOCHS = 10

LEARNING_RATE = 1e-3

NUM_CLASSES = 10


In [61]:
# DATASET PROCESSING

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data

# from torch.utils.data import DataLoader
from torch.utils.data import random_split


from torchvision import datasets
from torchvision import transforms

import time

train_data = datasets.MNIST(
    root=DATA_DIR,
    train=True,
    download=True
)

test_data = datasets.MNIST(
    root=DATA_DIR,
    train=False,
    download=True
)

n_train_examples = int(len(train_data) * TRAIN_RATIO)
n_valid_examples = len(train_data) - n_train_examples

train_data, valid_data = data.random_split(
    train_data,
    [n_train_examples, n_valid_examples]
)

mean = train_data.dataset.data.float().mean() / 255
std = train_data.dataset.data.float().std() / 255

train_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])

train_data.dataset.transform = train_transforms
valid_data.dataset.tranform = test_transforms

In [62]:
BATCH_SIZE = 256

train_dataloader = data.DataLoader(
    train_data,
    shuffle=True,
    batch_size=BATCH_SIZE
)

valid_dataloader = data.DataLoader(
    valid_data,
    batch_size=BATCH_SIZE
)

In [63]:
import torch.nn as nn
import torch.nn.functional as F

class LeNetClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels=1, 
            out_channels=6,
            kernel_size=5,
            padding="same"
        )
        self.avgpool1 = nn.AvgPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(
            in_channels=6, 
            out_channels=16,
            kernel_size=5
        )
        self.avgpool2 = nn.AvgPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc_1 = nn.Linear(16*5*5, 120)
        self.fc_2 = nn.Linear(120, 84)
        self.fc_3 = nn.Linear(84, num_classes)


    def forward(self, inputs):
        outputs = self.conv1(inputs)
        outputs = self.avgpool1(outputs)
        outputs = F.relu(outputs)
        outputs = self.conv2(outputs)
        outputs = self.avgpool2(outputs)
        outputs = F.relu(outputs)
        outputs = self.flatten(outputs)
        outputs = self.fc_1(outputs)
        outputs = self.fc_2(outputs)
        outputs = self.fc_3(outputs)

        return outputs

In [64]:
# Training function
def train ( model, optimizer, criterion, train_dataloader,
            device, epoch=0, log_interval=50):
    model.train()
    total_acc, total_count = 0, 0
    losses = []
    start_time = time.time()

    for idx, (inputs, labels) in enumerate(train_dataloader):
        optimizer.zero_grad()

        predictions = model(inputs)

        loss = criterion(predictions, labels)
        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),
        0.1)

        optimizer.step()
        total_acc += (predictions.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print(
                "| epoch {:3d} | {:5d}/{:5d} batches "
                "| accuracy {:8.3f} ".format(
                    epoch, idx, len(train_dataloader),
                    total_acc / total_count
                )
            )
            total_acc, total_count = 0, 0
            start_time = time.time()

        epoch_acc = total_acc / total_count
        epoch_loss = sum(losses) / len(losses)

        return epoch_acc, epoch_loss

In [65]:
# Evaluation 
def evaluate(model, criterion, valid_dataloader):
    model.eval()
    total_acc, total_count = 0, 0
    losses = []

    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(valid_dataloader):
            
            predictions = model(inputs)

            loss = criterion(predictions, labels)
            losses.append(loss.item())

            total_acc += (predictions.argmax(1) == labels).sum().item()
            total_count += labels.size(0)

    epoch_acc = total_acc / total_count
    epoch_loss = sum(losses) / len(losses)

    return epoch_acc, epoch_loss

In [66]:
num_classes = len(train_data.dataset.classes)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

lenet_model = LeNetClassifier(num_classes)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters())

num_epochs = 10

train_accs, train_losses = [], []
eval_accs, eval_losses = [], []

best_loss_eval = 100

In [67]:
for epoch in range(1, num_epochs+1):
    epoch_start_time = time.time()

    # Training
    train_acc, train_loss = train(lenet_model, optimizer, criterion, train_dataloader,
                            device, epoch)
    train_accs.append(train_acc)
    train_losses.append(train_loss)

    # Evaluation
    eval_acc, eval_loss = evaluate(lenet_model, criterion, valid_dataloader)
    eval_accs.append(eval_acc)
    eval_losses.append(eval_loss)

    # Save best model
    if eval_loss < best_loss_eval:
        torch.save(lenet_model.state_dict(), MODEL_PATH)

    # Print loss, acc end epoch
    print("-" * 60)
    print("| End of epoch {:3d} | Time : {:5.2f}s | Train Accuracy {:8.3f} | Train Loss {:8.3f} "
        "| Valid Accuracy {:8.3f} | Valid Loss {:8.3f} ".format(
            epoch, time.time() - epoch_start_time, train_acc, train_loss,
            eval_acc, eval_loss
        )
    )
    print("-" * 60)

    # Load best model
    lenet_model.load_state_dict(torch.load(MODEL_PATH))
    lenet_model.eval()


test_data.transform = test_transforms
test_dataloader = data.DataLoader(
    test_data,
    batch_size=BATCH_SIZE
)
test_acc, test_loss = evaluate(lenet_model, criterion, test_dataloader)

------------------------------------------------------------
| End of epoch   1 | Time :  0.36s | Train Accuracy    0.168 | Train Loss    2.300 | Valid Accuracy    0.172 | Valid Loss    2.285 
------------------------------------------------------------
------------------------------------------------------------
| End of epoch   2 | Time :  0.32s | Train Accuracy    0.160 | Train Loss    2.289 | Valid Accuracy    0.208 | Valid Loss    2.261 
------------------------------------------------------------
------------------------------------------------------------
| End of epoch   3 | Time :  0.31s | Train Accuracy    0.172 | Train Loss    2.265 | Valid Accuracy    0.420 | Valid Loss    2.234 
------------------------------------------------------------
------------------------------------------------------------
| End of epoch   4 | Time :  0.31s | Train Accuracy    0.430 | Train Loss    2.229 | Valid Accuracy    0.461 | Valid Loss    2.203 
---------------------------------------------